# BitRoss 2 · 2/3 · Caption with a vision-language model

Labels every kept texture with **Qwen2.5-VL-3B-Instruct** through vLLM: a short name, a one-line
caption, a validity flag (drops UI pieces / fragments the heuristics missed) and a 1-5 quality score.
Shards are written to Drive as they finish, so you can stop and resume any time.

**Runtime:** L4 GPU. Throughput is roughly 40-80 images/s → ~3-4 h for ~650K images.
Images with meaningless file names are labeled first, so even a partial run helps a lot.

In [ ]:
#@title Setup: Drive, code, Hugging Face login
#@markdown Put your Hugging Face token in Colab **Secrets** (key icon in the left bar) as `HF_TOKEN`
#@markdown and enable notebook access. The dataset is private, so the token is required.
REPO_URL = "https://github.com/OVAWARE/BitRoss"  #@param {type:"string"}
BRANCH = "v2"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/BitRoss2"  #@param {type:"string"}

import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists("/content/BitRoss"):
    code_zip = f"{DRIVE_ROOT}/code.zip"
    if os.path.exists(code_zip):            # fallback: upload a zip of the repo to Drive
        subprocess.run(["unzip", "-q", code_zip, "-d", "/content/BitRoss"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/BitRoss"], check=True)
os.chdir("/content/BitRoss")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title Install vLLM and bring the curated work dir from Drive
!pip -q install vllm qwen-vl-utils
!mkdir -p /content/work && cp {DRIVE_ROOT}/work/pixels.npy {DRIVE_ROOT}/work/filtered.parquet {DRIVE_ROOT}/work/palette.json {DRIVE_ROOT}/work/scan.parquet /content/work/

In [ ]:
#@title Quick check on 200 images before the long run
!python -m bitross.data.vlm_label --work /content/work --out /content/label_test --limit 200 --shard-size 200 --batch 200
!head -n 15 /content/label_test/labels_0000.jsonl

In [ ]:
#@title Full run (resumable; re-run this cell after a disconnect)
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"  #@param {type:"string"}
!python -m bitross.data.vlm_label --work /content/work --out {DRIVE_ROOT}/labels --model {MODEL} --batch 512

In [ ]:
#@title Build the training cache with VLM captions merged in
!python -m bitross.data.build cache --work /content/work --out /content/cache --vlm-labels "{DRIVE_ROOT}/labels/*.jsonl"
!mkdir -p {DRIVE_ROOT}/cache && cp -r /content/cache/* {DRIVE_ROOT}/cache/